# Vegetation challenge: data exploration

Step 1: load the files and see what we have. Keep printed output to summaries (counts, column names). Do not print raw rows, because the data are not published.

In [18]:
from pathlib import Path

import pandas as pd

## Paths

The data live outside the repository. Change `data_dir` if the files move or you are someone reproducing this workflow from your own machine.

In [19]:
data_dir = Path("/Users/scohen/Documents/NaturalState/Take Home Project/Data Vegetation")

exports_dir = data_dir / "ODK Data Exports"
entities_dir = data_dir / "Entity lists"

print(exports_dir.exists(), entities_dir.exists())

True True


## Read the files

The ODK exports are the data to check. The entity lists are the reference data.

In [20]:
exports = {p.stem: pd.read_csv(p) for p in sorted(exports_dir.glob("*.csv"))}
entities = {p.stem: pd.read_csv(p) for p in sorted(entities_dir.glob("*.csv"))}

## Overview: rows and columns per file

In [21]:
def overview(tables):
    return pd.DataFrame(
        {"rows": {name: df.shape[0] for name, df in tables.items()},
         "columns": {name: df.shape[1] for name, df in tables.items()}}
    )

overview(exports)

,rows,columns
herbaceous_veg_survey-additional_species_repeat,1296,30
herbaceous_veg_survey-quadrat_repeat,640,21
herbaceous_veg_survey,32,52
register_vegetation_plots,31,82


In [22]:
overview(entities)

,rows,columns
areas,1,10
centroids,51,21
ecosystem_types,5,10
project_team,6,13
projects,1,11
species,370,21
species_extra,250,14
species_extra_ids,109,11
surveys,4,15
transport,5,10


### Load data tables

In [23]:
for name, df in {**exports, **entities}.items():
    globals()[name.replace("-", "_")] = df

### herbaceous_veg_survey

This is the main survey file. Each row is one survey submission: a field team's visit to one registered vegetation plot, recorded in ODK using the herbaceous vegetation form. The columns describe who did the survey (team and recorder), when, which plot and survey it belongs to, how many quadrats were recorded, and where the submission was made. The species found in each of the 20 quadrats are in the linked `quadrat_repeat` and `additional_species_repeat` files, which join back to this file through `KEY`. The data are used to assess herbaceous species richness and diversity.

In [24]:
form = pd.read_excel(data_dir / "ODK Data Collection Forms" / "herbaceous_veg_survey.xlsx", sheet_name="survey")
form = form.dropna(subset=["type"])

# Walk through the form, building each question's column name from its groups
groups, in_repeat, rows = [], 0, []
for _, q in form.iterrows():
    kind = str(q["type"]).strip()
    if kind == "begin_group":
        groups.append(q["name"])
    elif kind == "end_group":
        groups.pop()
    elif kind == "begin_repeat":
        in_repeat += 1
    elif kind == "end_repeat":
        in_repeat -= 1
    elif in_repeat == 0:          # questions inside repeats belong to the other files
        text = q["label"] if pd.notna(q["label"]) else "calculated: " + str(q["calculation"])
        rows.append({"column": "-".join(groups + [q["name"]]), "form_type": kind,
                     "description": str(text)[:100]})
form_cols = pd.DataFrame(rows)

# Location questions are exported as four columns
geo = form_cols[form_cols["form_type"].isin(["geopoint", "background-geopoint"])]
form_cols = pd.concat([form_cols] + [geo.assign(column=geo["column"] + "-" + s)
                      for s in ["Latitude", "Longitude", "Altitude", "Accuracy"]], ignore_index=True)

survey = herbaceous_veg_survey
schema = (pd.DataFrame({"column": survey.columns, "dtype": survey.dtypes.astype(str).values,
                        "missing": survey.isna().sum().values, "distinct": survey.nunique().values})
          .merge(form_cols, on="column", how="left"))
schema["form_type"] = schema["form_type"].fillna("ODK system column")
schema

/Users/scohen/Documents/GitHub/NS_SDS_TakeHomeProject_SCohen/.venv/lib/python3.13/site-packages/openpyxl/worksheet/_reader.py:329: UserWarning: Data Validation extension is not supported and will be removed
  warn(msg)


,column,dtype,missing,distinct,form_type,description
0,SubmissionDate,str,0,32,ODK system column,NaN
1,note_overview,float64,32,0,note,This form is to be used by field data teams to...
2,field_team_specifics-selected_transport_type_uuid,str,0,1,select_one transport.csv,Select the the mode of transport:
3,field_team_specifics-transport_type_label,str,0,1,calculate,"calculated: instance(""transport"")/root/item[na..."
4,field_team_specifics-selected_project_team_uui...,str,0,2,select_multiple_from_file project_team.csv,Select all the team members involved in this c...
5,field_team_specifics-recorder_uuid,str,0,2,select_one_from_file project_team.csv,Who is recording the data:
6,field_team_specifics-recorder_label,str,0,2,calculate,"calculated: instance(""project_team"")/root/item..."
7,survey_begin-start_time,str,0,32,start,Start time
8,survey_begin-survey_year,int64,0,1,calculate,"calculated: format-date(${start_time}, '%Y')"
9,survey_begin-selected_survey_uuid,str,0,1,select_one_from_file surveys.csv,Select a Survey:


### Columns kept for the checks

`herbaceous_veg_survey` has 52 columns. I kept 22 of them: the ones needed to check who surveyed which plot, when, how many quadrats, where, and whether the submission is complete.

**What I kept, and the checks each group supports**

| Group | Why it is kept |
|---|---|
| Keys and links (`KEY`, plot columns) | Find repeated plot visits and confirm the plot is registered and viable |
| Team (recorder and team members) | Confirm the recorder is a valid team member |
| Timing (`SubmissionDate`, start and end time) | Check dates and survey duration |
| Survey context (survey, year, replicate) | Check the survey matches the plot's assigned survey |
| Sampling effort (quadrat count, quadrats with species) | Check for exactly 20 quadrats per plot |
| Location (latitude, longitude, accuracy) | Check the location is present, accurate, and near the plot |
| Completeness (attachments, review state, confirmation) | Check for missing photos and review flags |

**What I dropped**

- **Columns that never vary.** The project, area, survey type, target group, and transport labels each have a single value in all 32 rows, so they cannot show a problem.
- **Notes that are always empty.** The `note_*` and `quadrat_check_*` columns and the other note and heading fields are blank in every row.
- **ODK internals.** `base_active_herb_count`, `new_unknown_index_list`, `new_missing_index_list`, `meta-*`, `SubmitterID`, `Edits`, `FormVersion` and `Status` are system or calculation columns, not field observations. `DeviceID` is also dropped but could be kept to compare devices.

**Two things I noticed while choosing, to check later**

- There are 32 rows but only 30 distinct plots, so some plots were surveyed more than once.
- 3 submissions have no location (`background_geopoint` is empty).

In [25]:
keep = [
    "KEY", "plot_selection-selected_plot_uuid", "plot_selection-plot_name", "plot_selection-get_plot_status",
    "field_team_specifics-selected_project_team_uuid_multi", "field_team_specifics-recorder_uuid", "field_team_specifics-recorder_label",
    "SubmissionDate", "survey_begin-start_time", "survey_end-end_time",
    "survey_begin-selected_survey_uuid", "survey_begin-survey_year", "survey_begin-replicate",
    "observations-quadrat_repeat_count", "survey_end-quadrats_with_species",
    "survey_end-background_geopoint-Latitude", "survey_end-background_geopoint-Longitude", "survey_end-background_geopoint-Accuracy",
    "AttachmentsPresent", "AttachmentsExpected", "ReviewState", "survey_end-confirm_submission",
]

herbaceous_veg_survey_keep = herbaceous_veg_survey[keep]

### herbaceous_veg_survey-quadrat_repeat

This file holds the quadrat records. Each row is one quadrat (a 1 m × 1 m frame placed along the plot's transect) within one survey submission. Each survey should have 20 quadrats, so the 32 submissions give 640 rows. For each quadrat, the field team recorded whether any herbaceous species were present, which species from the approved list were selected, whether additional species were present, a photo, the GPS location, and an optional comment. It links to its survey through `PARENT_KEY` (matching the survey's `KEY`), and its species selections link to the `species` entity list. The species recorded here are the basis for assessing species richness and diversity.


In [27]:
def form_columns(form_path, repeat=None):
    form = pd.read_excel(form_path, sheet_name="survey").dropna(subset=["type"])
    stack, rows = [], []
    for _, q in form.iterrows():
        kind, name = str(q["type"]).strip(), q["name"]
        if kind == "begin_group":
            stack.append(("group", name)); continue
        if kind == "begin_repeat":
            stack.append(("repeat", name)); continue
        if kind in ("end_group", "end_repeat"):
            stack.pop(); continue
        repeats = [i for i, (t, n) in enumerate(stack) if t == "repeat"]
        innermost = stack[repeats[-1]][1] if repeats else None
        if innermost != repeat:
            continue
        start = repeats[-1] + 1 if repeats else 0
        groups = [n for t, n in stack[start:] if t == "group"]
        text = q["label"] if pd.notna(q["label"]) else "calculated: " + str(q["calculation"])
        rows.append({"column": "-".join(groups + [name]), "form_type": kind, "description": str(text)[:100]})
    cols = pd.DataFrame(rows)
    geo = cols[cols["form_type"].isin(["geopoint", "background-geopoint"])]
    return pd.concat([cols] + [geo.assign(column=geo["column"] + "-" + s)
                     for s in ["Latitude", "Longitude", "Altitude", "Accuracy"]], ignore_index=True)

def export_schema(df, form_cols):
    schema = pd.DataFrame({"column": df.columns, "dtype": df.dtypes.astype(str).values,
                           "missing": df.isna().sum().values, "distinct": df.nunique().values})
    schema = schema.merge(form_cols, on="column", how="left")
    schema["form_type"] = schema["form_type"].fillna("ODK system column")
    return schema


survey_form = data_dir / "ODK Data Collection Forms" / "herbaceous_veg_survey.xlsx"

quadrat_schema = export_schema(
    herbaceous_veg_survey_quadrat_repeat,
    form_columns(survey_form, "quadrat_repeat"),
)
quadrat_schema

/Users/scohen/Documents/GitHub/NS_SDS_TakeHomeProject_SCohen/.venv/lib/python3.13/site-packages/openpyxl/worksheet/_reader.py:329: UserWarning: Data Validation extension is not supported and will be removed
  warn(msg)


,column,dtype,missing,distinct,form_type,description
0,quadrat_number,int64,0,20,calculate,Quadrat Number
1,note_new_quadrat,float64,640,0,note,Move to **Quadrat ${quadrat_number}**
2,herbs_present,str,0,2,select_one yes_no,Are there any herbaceous species present?
3,herb_species-all_unknown_unidentifiable,str,7,2,select_one yes_no,Are **ALL** of the specimens either unidentifi...
4,herb_species-selected_herb_species_uuids,str,146,225,select_multiple species.csv,Select **ALL** of the species present:
5,herb_species-count_herb_species,float64,7,7,calculate,calculated: count-selected(${selected_herb_spe...
6,herb_species-note_herb_species_count,float64,640,0,note,**${count_herb_species}** species selected\n
7,additional_species_present,str,7,2,select_one yes_no,Are there any species present in the quadrat t...
8,species_list-known_species_list,str,146,188,calculate,calculated: if(${selected_herb_species_uuids} ...
9,species_list-additional_species_list,str,66,409,calculate,"calculated: join('<br/>', ${validated_name}[. ..."


### Columns kept for the checks: quadrat_repeat

`herbaceous_veg_survey-quadrat_repeat` has 21 columns. I kept 14: the ones needed to check that each survey has its 20 quadrats, that species and photos were recorded, and that locations are accurate.

**What I kept, and the checks each group supports**

| Columns | Why it is kept |
|---|---|
| `KEY`, `PARENT_KEY` | Identify each quadrat and link it to its survey |
| `quadrat_number` | Each survey should have quadrats 1 to 20, once each |
| `herbs_present`, `herb_species-all_unknown_unidentifiable` | Find quadrats with no species recorded, or where nothing could be matched to the species list |
| `herb_species-selected_herb_species_uuids`, `herb_species-count_herb_species` | The species recorded, and a cross-check that the count matches the number selected |
| `additional_species_present`, `species_list-all_species_list` | Link to the extra-species file, and a readable combined species list |
| `quadrat_image` | Check that a photo was taken for each quadrat |
| Latitude, longitude, accuracy | Check that the location is present, accurate (5 m or better per the SOP), and on the transect |
| `quadrat_comment` | Notes on unusual conditions (only a few are filled in) |

**What I dropped**

- **Empty notes.** `note_new_quadrat`, `herb_species-note_herb_species_count`, `species_list-note_species_summary_heading` and `species_list-note_species_list` are blank in all 640 rows.
- **Display-only lists.** `species_list-known_species_list` and `species_list-additional_species_list` are text the form builds from other columns, so they can be recreated from the species selections and the extra-species file.
- **Altitude.** The SOP sets no rule for altitude, so `location_quadrat-Altitude` supports no check.

In [28]:
quadrat_keep_cols = [
    "KEY", "PARENT_KEY", "quadrat_number",
    "herbs_present", "herb_species-all_unknown_unidentifiable",
    "herb_species-selected_herb_species_uuids", "herb_species-count_herb_species",
    "additional_species_present", "species_list-all_species_list",
    "quadrat_image",
    "location_quadrat-Latitude", "location_quadrat-Longitude", "location_quadrat-Accuracy",
    "quadrat_comment",
]

herbaceous_veg_survey_quadrat_repeat_keep = herbaceous_veg_survey_quadrat_repeat[quadrat_keep_cols]

herbaceous_veg_survey_quadrat_repeat_keep.shape   # should be (640, 14)

(640, 14)

### herbaceous_veg_survey-additional_species_repeat

This file holds the additional species records. Each row is one species recorded in a quadrat that was not on the approved species list, could not be identified in the field, or was an unknown or missing species already recorded earlier in the same submission. The 1,296 rows come from 574 of the 640 quadrats; the other 66 quadrats had no additional species. The `species_entry_mode` column shows how each species was recorded: as a new unknown (`new_unknown`), a new missing species typed in by name (`new_missing`), or a reused unknown or missing species (`reuse_unknown`, `reuse_missing`). It links to its quadrat through `PARENT_KEY` (matching the quadrat's `KEY`), and the reused species link to the `species_extra` entity list. New records are written to `species_extra` with a pending review status, so Natural State can confirm the identification. These records matter for richness because unknown and missing species are recorded under provisional names, and how to count them needs to be decided.

In [ ]:
herbaceous_veg_survey_additional_species_repeat_schema = export_schema(
    herbaceous_veg_survey_additional_species_repeat,
    form_columns(survey_form, "additional_species_repeat"),
)
herbaceous_veg_survey_additional_species_repeat_schema

### Columns kept for the checks: additional_species_repeat

`herbaceous_veg_survey-additional_species_repeat` has 30 columns. I kept 12: the ones needed to check that each additional species is linked to a quadrat, was recorded with a valid name, and has the photo and review status the form requires.

**What I kept, and the checks each group supports**

| Columns | Why it is kept |
|---|---|
| `KEY`, `PARENT_KEY` | Identify each record and link it to its quadrat |
| `observation_index` | Position of the record within its quadrat |
| `species_entry_mode` | How the species was recorded, which decides which other columns should be filled |
| `select_reuse_unknown`, `select_reuse_missing` | Link to `species_extra`, to check each reused species exists |
| `new_missing_canonical` | The species name typed in the field, to check its format (`Genus_species`) and spelling |
| `validated_name` | The final species name for the record, used to count species. It should never be empty |
| `entity_label_to_write`, `review_status`, `new_record_reason` | The new records written to `species_extra`, and whether they are still awaiting review |
| `macro_images-macro_1` | The close-up photo, which the form requires for every new unknown species |

**What I dropped**

- **Empty notes.** `note_new_unknown`, `note_reuse_unknown`, `note_new_missing`, `note_reuse_missing` and `macro_images-note_images` are blank in all 1,296 rows.
- **Calculation helpers.** `count_prior_new_in_submission`, `assigned_unknown_index` and `target_extra_name` are counters the form uses to assign new records. `auto_new_unknown_label`, `reused_unknown_label`, `auto_new_missing_label` and `reused_missing_label` are display labels that can be recreated from `species_extra`.
- **Constant or duplicate columns.** `write_choice_status` is always `active`, and `meta-entity-label` is identical to `entity_label_to_write`.
- **Optional extra photos.** `macro_images-macro_2` to `macro_images-macro_5` are optional, and three of the four are empty.

**One thing I noticed while choosing, to check later**

- `validated_name` is empty in 1 of the 1,296 rows.

In [ ]:
additional_species_repeat_keep_cols = [
    "KEY", "PARENT_KEY", "observation_index",
    "species_entry_mode", "select_reuse_unknown", "select_reuse_missing",
    "new_missing_canonical", "validated_name",
    "entity_label_to_write", "review_status", "new_record_reason",
    "macro_images-macro_1",
]

herbaceous_veg_survey_additional_species_repeat_keep = herbaceous_veg_survey_additional_species_repeat[additional_species_repeat_keep_cols]

herbaceous_veg_survey_additional_species_repeat_keep.shape   # should be (1296, 12)

### register_vegetation_plots

This file holds the plot registrations. Each row is one registration of a vegetation plot (a 50 m × 5 m belt transect) by a field team, using the plot registration form. The 31 rows are 31 distinct plots. The team picks a planned sampling point from the `centroids` list and decides whether the plot is viable. If it is, they record its ecosystem type, woody vegetation, soil, ground cover and disturbance, plus photos and GPS locations at endpoint A, the midpoint and endpoint B. Registration creates the plot's row in the `vegplots` entity list, which the survey form's plot picker then draws from, and updates the matching centroid. In this export 30 plots are viable and 1 is not, and the non-viable plot has no metadata or locations. It links to `centroids` through `plot_selection-selected_plot_uuid`, to `surveys` through `survey_begin-selected_survey_uuid_multi`, and to `ecosystem_types` through `plot_metadata-selected_ecosystem_type_uuid`.

In [ ]:
register_form = data_dir / "ODK Data Collection Forms" / "register_vegetation_plots.xlsx"

register_vegetation_plots_schema = export_schema(
    register_vegetation_plots,
    form_columns(register_form),
)
register_vegetation_plots_schema

### Columns kept for the checks: register_vegetation_plots

`register_vegetation_plots` has 82 columns. I kept 28: the ones needed to check who registered which plot and when, that the plot comes from a planned sampling point, and that the transect locations are present and consistent with the SOP.

**What I kept, and the checks each group supports**

| Columns | Why it is kept |
|---|---|
| `KEY`, `SubmissionDate`, start and end time | Identify each registration, and check dates and duration |
| Team (recorder and team members) | Confirm the recorder is a valid team member |
| `survey_begin-selected_survey_uuid_multi` | Which surveys the plot is assigned to |
| `selected_plot_uuid`, `plot_name`, `geohash`, `sample_status` | Plot comes from a planned centroid, is primary or backup, and its midpoint matches the geohash |
| `is_plot_viable`, `plot_establishment_notes` | A non-viable plot must have a reason |
| `selected_ecosystem_type_uuid`, `disturbance_types` | Valid ecosystem type, and disturbance such as fire that makes a plot non-viable |
| Latitude, longitude and accuracy at endpoint A, the midpoint and endpoint B | Transect length (50 m), orientation, midpoint within 25 m of the planned point, and accuracy of 5 m or better |
| `AttachmentsPresent`, `AttachmentsExpected`, `confirm_submission` | Missing photos, and confirmed submission |

**What I dropped**

- **Empty notes.** `note_overview`, `survey_begin-note_parent_labels`, `transect_orientation_note`, `survey_end-note_field_submission` and `survey_end-plot_establishment_notes` are blank in all 31 rows.
- **Columns that never vary or are copied from other lists.** Transport, default surveys, survey year, survey label, project, area, baseline year, replicate, stratum and plot label are the same in every row or are looked up from the `surveys` and `centroids` lists.
- **Habitat description.** The woody species, canopy, soil, ground cover and water questions describe the habitat, but the SOP sets no rule for them and this task is about herbaceous vegetation. `ecosystem_type_name` is a lookup of the ecosystem id I kept.
- **Photos, altitude and joined locations.** The four individual photo columns are replaced by the attachment counts. The SOP sets no rule for altitude. `vegplot_transect` is the three locations joined into one text. `background_geopoint` is empty in 16 of the 31 rows and is not needed because the transect points are recorded separately.
- **ODK internals.** `update_vegplot_link`, `update_marker_color`, the `meta-*` columns, `SubmitterID`, `SubmitterName`, `Status`, `ReviewState`, `DeviceID`, `Edits` and `FormVersion` are system columns, and `Status` and `ReviewState` are empty.

**Things I noticed while choosing, to check later**

- 30 plots are viable and 1 is not. The non-viable plot has no ecosystem, metadata or locations.
- 30 plots are primary sampling points and 1 is a backup.

In [29]:
register_vegetation_plots_keep_cols = [
    "KEY", "SubmissionDate", "survey_begin-start_time", "survey_end-end_time",
    "field_team_specifics-recorder_uuid", "field_team_specifics-recorder_label",
    "field_team_specifics-selected_project_team_uuid_multi",
    "survey_begin-selected_survey_uuid_multi",
    "plot_selection-selected_plot_uuid", "plot_selection-plot_name",
    "plot_selection-geohash", "plot_selection-sample_status",
    "plot_selection-is_plot_viable", "plot_selection-plot_establishment_notes",
    "plot_metadata-selected_ecosystem_type_uuid", "plot_metadata-disturbance_types",
    "plot_metadata-collect_points-endpoint_a-location_endpoint_a-Latitude",
    "plot_metadata-collect_points-endpoint_a-location_endpoint_a-Longitude",
    "plot_metadata-collect_points-endpoint_a-location_endpoint_a-Accuracy",
    "plot_metadata-collect_points-midpoint-location_midpoint-Latitude",
    "plot_metadata-collect_points-midpoint-location_midpoint-Longitude",
    "plot_metadata-collect_points-midpoint-location_midpoint-Accuracy",
    "plot_metadata-collect_points-endpoint_b-location_endpoint_b-Latitude",
    "plot_metadata-collect_points-endpoint_b-location_endpoint_b-Longitude",
    "plot_metadata-collect_points-endpoint_b-location_endpoint_b-Accuracy",
    "AttachmentsPresent", "AttachmentsExpected", "survey_end-confirm_submission",
]

register_vegetation_plots_keep = register_vegetation_plots[register_vegetation_plots_keep_cols]

register_vegetation_plots_keep.shape   # should be (31, 28)

(31, 28)

### vegplots

This is an entity list, a reference table rather than field data. Each row is one registered vegetation plot, created when the field team submits the plot registration form. The 31 rows match the 31 registrations. The survey form's plot picker draws from this list and offers only plots that belong to the selected survey and are marked viable. Each row records the plot's name, the geohash of its midpoint, whether it is a primary or backup sampling point, whether it is viable, the midpoint location (`geometry`: latitude, longitude, altitude and accuracy), and the two surveys it belongs to. The survey file's `plot_selection-selected_plot_uuid` matches `__id` here, and `plot_uuid` matches `__id` in the `centroids` list, the planned sampling point. The columns that start with `__` are added by ODK Central.

In [ ]:
def table_profile(df):
    return pd.DataFrame({"column": df.columns, "dtype": df.dtypes.astype(str).values,
                         "missing": df.isna().sum().values, "distinct": df.nunique().values,
                         "unique": [df[c].is_unique and df[c].notna().all() for c in df.columns]})


vegplots_profile = table_profile(vegplots)
vegplots_profile

### Columns kept for the checks: vegplots

`vegplots` has 19 columns. I kept 10: the ones needed to check that surveyed plots are registered, viable and assigned to the right survey, and to compare each plot's midpoint with its planned location.

**What I kept, and the checks each group supports**

| Columns | Why it is kept |
|---|---|
| `__id` | The key the survey file uses to pick a plot |
| `plot_uuid` | Link to the planned sampling point in `centroids` |
| `plot_name`, `geohash` | Identify the plot, and check its midpoint against the geohash cell |
| `plot_status` | Primary or backup sampling point |
| `is_viable` | The survey form offers only viable plots, so a survey of a non-viable plot is a problem |
| `geometry` | The midpoint location (latitude, longitude, altitude and accuracy) |
| `survey_uuids` | The surveys the plot is assigned to, to check a survey matches its plot |
| `__createdAt`, `__creatorName` | When the plot was registered and by which team |

**What I dropped**

- **Columns that never vary.** `replicate`, `survey_labels`, `area_label` and `stratum_label` have a single value in all 31 rows, and `__updates` and `__version` do too.
- **Duplicates and empty columns.** `label` is identical to `plot_name`, and `__updatedAt` is empty in every row.
- **ODK internals.** `__creatorId` is an id for the team already shown by `__creatorName`.

**Things I noticed while choosing, to check later**

- 30 plots are viable and 1 is not. The non-viable plot is a primary plot, and it is the only row with no `geometry`.
- 30 plots are primary and 1 is a backup, and the backup is a different plot from the non-viable one.
- Every plot belongs to 2 surveys.

In [ ]:
vegplots_keep_cols = [
    "__id", "plot_uuid", "plot_name", "geohash", "plot_status",
    "is_viable", "geometry", "survey_uuids", "__createdAt", "__creatorName",
]

vegplots_keep = vegplots[vegplots_keep_cols]

vegplots_keep.shape   # should be (31, 10)

### centroids

This is an entity list of the planned sampling points. Each row is one centroid: the location a field team navigates to and uses as the midpoint of a plot. There are 51 centroids, 36 primary and 15 backup, in two strata (savanna and shrubland). During plot registration the team picks a centroid from this list. The registration then creates a row in `vegplots` and marks the centroid as linked (`vegplot_linked` is `yes`). 31 centroids are linked, matching the 31 registered plots, and 20 have not been registered. The `geometry` column holds the planned location as four space-separated values: latitude, longitude, altitude and accuracy. All locations use the coordinate system EPSG:4326 (latitude and longitude). `choice_name` is the plot name, and `label` is that name with the status added, such as `(backup)`. `vegplots.plot_uuid` matches `__id` here. The columns that start with `__` are added by ODK Central.

In [ ]:
centroids_profile = table_profile(centroids)
centroids_profile

### Columns kept for the checks: centroids

`centroids` has 21 columns. I kept 7: the ones needed to compare each registered plot with its planned location, and to find planned points that have not been registered.

**What I kept, and the checks each group supports**

| Columns | Why it is kept |
|---|---|
| `__id` | The key that `vegplots.plot_uuid` and the registration form point to |
| `choice_name` | The plot name, to match `vegplots` and the survey data |
| `geohash`, `geometry` | The planned location of the plot's midpoint, to compare with the registered midpoint (it should be within 25 m) |
| `sample_status` | Primary or backup sampling point |
| `stratum_label` | The habitat stratum the point was drawn from |
| `vegplot_linked` | Whether the point has been registered, to find planned points with no plot |

**What I dropped**

- **Columns that never vary.** `project_label`, `project_abbr`, `project_area_name`, `crs` and `replicate` have a single value in all 51 rows. The coordinate system is EPSG:4326 for every row.
- **Duplicates and display columns.** `label` is the plot name plus its status, which are already in `choice_name` and `sample_status`. `marker-color` is the colour the point shows on the form's map.
- **Mostly empty.** `survey_year` is empty for the 19 points that were never registered.
- **ODK internals.** `__createdAt`, `__creatorId`, `__creatorName`, `__updates`, `__updatedAt` and `__version` are system columns.

**Things I noticed while choosing, to check later**

- 31 of the 51 planned points are registered and 20 are not (6 primary and 14 backup).
- `vegplot_linked` is `yes` for exactly the 31 centroids that appear in `vegplots`.

In [ ]:
centroids_keep_cols = [
    "__id", "choice_name", "geohash", "geometry",
    "sample_status", "stratum_label", "vegplot_linked",
]

centroids_keep = centroids[centroids_keep_cols]

centroids_keep.shape   # should be (51, 7)

### species

This is an entity list of the approved species for the project. Each row is one plant species, 370 in total: 238 herbaceous and 132 woody. The `groups` column says which, and the survey form offers only the `herbaceous` species in its species picker. The survey file's species selections (`herb_species-selected_herb_species_uuids`) match `__id` here. `label` is the species name without authorship, which is the name shown on the form, and `scientific_name` adds the authority. The other columns give the taxonomy (family, genus, and so on) and where the name came from. The columns that start with `__` are added by ODK Central.

In [ ]:
species_profile = table_profile(species)
species_profile

### Columns kept for the checks: species

`species` has 21 columns. I kept 6: the ones needed to check that recorded species are on the approved list and are herbaceous, and to summarize species by family and genus.

**What I kept, and the checks each group supports**

| Columns | Why it is kept |
|---|---|
| `__id` | The key the quadrat file uses for each selected species |
| `label`, `scientific_name` | The species name, to report which species were recorded |
| `groups` | Herbaceous or woody, because only herbaceous species should be recorded here |
| `family`, `genus` | To summarize richness and diversity by family or genus |

**What I dropped**

- **Empty or constant columns.** `vernacular_name` and `global_taxon_id` are empty in all 370 rows, and `rank`, `kingdom` and `phylum` have a single value.
- **Detail not needed for the checks.** `class`, `order`, `sources` and `gbif_taxon_key` describe the taxonomy and where each name came from.
- **ODK internals.** `__createdAt`, `__creatorId`, `__creatorName`, `__updates`, `__updatedAt` and `__version` are system columns.

**Thing I noticed while choosing, to check later**

- The approved list has 132 woody species. A woody species recorded in a herbaceous quadrat would be a problem.

In [ ]:
species_keep_cols = ["__id", "label", "scientific_name", "family", "genus", "groups"]

species_keep = species[species_keep_cols]

species_keep.shape   # should be (370, 6)

### species_extra

This is an entity list of the species added in the field. Each row is one record the form can use for an unknown species (one that could not be identified) or a missing species (one not on the approved list). Of the 250 rows, 200 are herbaceous and 50 are woody. Many rows are inactive (141): the form holds them in reserve and activates the next one when a new species is recorded, which also sets its review status to `pending` and gives the reason (`unknown_species` or `missing_species`). The additional species file's `select_reuse_unknown` and `select_reuse_missing` match `__id` here. The columns that start with `__` are added by ODK Central.

In [ ]:
species_extra_profile = table_profile(species_extra)
species_extra_profile

### Columns kept for the checks: species_extra

`species_extra` has 14 columns. I kept 7: the ones needed to check that new and reused species exist, are active, and are waiting for review.

**What I kept, and the checks each group supports**

| Columns | Why it is kept |
|---|---|
| `__id` | The key the additional species file uses for reused species |
| `label` | The species name, or the placeholder name for an unknown |
| `choice_status` | Active or inactive, to check that records in use are active |
| `unknown_index` | The number the form assigns to each new record |
| `review_status`, `new_record_reason` | Whether a new record is waiting for review, and why it was added |
| `group_type` | Herbaceous or woody |

**What I dropped**

- **Empty column.** `new_assigned_uuid` is empty in all 250 rows. It would hold the identified species once a record is resolved.
- **ODK internals.** `__createdAt`, `__creatorId`, `__creatorName`, `__updates`, `__updatedAt` and `__version` are system columns.

**Things I noticed while choosing, to check later**

- `label` has 249 distinct values in 250 rows, so one label appears twice.
- 109 records are active and all of them are pending review. 104 of the active records are herbaceous.
- The additional species file has 107 new records (62 new unknown and 45 new missing), compared with 104 active herbaceous records here, so these are worth reconciling.

In [ ]:
species_extra_keep_cols = [
    "__id", "label", "choice_status", "unknown_index",
    "review_status", "new_record_reason", "group_type",
]

species_extra_keep = species_extra[species_extra_keep_cols]

species_extra_keep.shape   # should be (250, 7)

### project_team

This is an entity list of the people who can be chosen as team members or recorder on the forms. Each row is one person, 6 in total, all active. Five belong to Acacia Trust and one to Natural State. The survey and registration files' recorder and team member columns match `__id` here. The columns that start with `__` are added by ODK Central.

In [ ]:
project_team_profile = table_profile(project_team)
project_team_profile

### Columns kept for the checks: project_team

`project_team` has 13 columns. I kept 5: the ones needed to check that the recorder and team members on each submission are valid and active.

**What I kept, and the checks each group supports**

| Columns | Why it is kept |
|---|---|
| `__id` | The key the survey and registration files use for the recorder and team members |
| `label`, `choice_name` | The name shown on the form, and the person's full name |
| `choice_status` | Whether the person is active, because the form offers only active people |
| `affiliation` | The organization, to summarize who did the work |

**What I dropped**

- **Name parts.** `first_name` and `family_name` are already combined in `choice_name`.
- **ODK internals.** `__createdAt`, `__creatorId`, `__creatorName`, `__updates`, `__updatedAt` and `__version` are system columns.

In [ ]:
project_team_keep_cols = ["__id", "label", "choice_name", "choice_status", "affiliation"]

project_team_keep = project_team[project_team_keep_cols]

project_team_keep.shape   # should be (6, 5)